In [ ]:
import os
import sys
from pathlib import Path

# Run from the Transformixer project root so outputs/checkpoints resolve correctly.
TRANSFORMIXER_ROOT = (Path.cwd().parent / "models" / "transformixer").resolve()
os.chdir(TRANSFORMIXER_ROOT)
sys.path.insert(0, str(TRANSFORMIXER_ROOT))

In [13]:
import wandb
from lightning.pytorch.callbacks import StochasticWeightAveraging

from xlstm_mixer.cli_helper import LoggerSaveConfigCallback, TaskCLI
from xlstm_mixer.exp.exp import ForecastingExp
from xlstm_mixer.lit.data import TSLibDataModule

# User-specified arguments; everything else uses TaskCLI / LightningCLI defaults.
dataset = "Traffic"
pred_len = 96
seq_len = 96
lr = 0.0005
batch_size = 32
d_model = 1024
e_layers = 2
n_heads = 16
dropout = 0.1
gamma = 0.99
cosine_epochs = 5
warmup_epochs = 2
constant_gamma_epochs = 1
seed = 42
max_epochs = 10
fast_dev_run = False
root_path = "../../datasets"

rest_args = [
    "--data", "ForecastingTSLibDataModule",
    "--data.dataset_name", dataset,
    "--data.root_path", root_path,
    "--optimizer.lr", str(lr),
    "--data.seq_len", str(seq_len),
    "--data.pred_len", str(pred_len),
    "--data.label_len", "0",
    "--data.batch_size", str(batch_size),
    "--data.num_workers", "4",
    "--data.persistent_workers", "true",
    "--model", "LongTermForecastingExp",
    "--model.criterion", "torch.nn.L1Loss",
    "--model.architecture", "Transformixer",
    "--model.architecture.n_heads", str(n_heads),
    "--model.architecture.e_layers", str(e_layers),
    "--model.architecture.d_model", str(d_model),
    "--model.architecture.dropout", str(dropout),
    "--lr_scheduler.constant_gamma_epochs", str(constant_gamma_epochs),
    "--lr_scheduler.gamma", str(gamma),
    "--lr_scheduler.cosine_epochs", str(cosine_epochs),
    "--lr_scheduler.warmup_epochs", str(warmup_epochs),
    "--trainer.logger.name", f"{dataset}_transformixer_{pred_len}_{seed}",
    "--trainer.logger.project", "transformixer",
    "--trainer.max_epochs", str(max_epochs),
    "--seed_everything", str(seed),
    "--trainer.fast_dev_run", str(fast_dev_run).lower(),
]

cli = TaskCLI(
    ForecastingExp,
    TSLibDataModule,
    subclass_mode_data=True,
    subclass_mode_model=True,
    run=False,
    args=rest_args,
    save_config_callback=LoggerSaveConfigCallback,
)
cli.datamodule.root_path = Path(root_path)
cli.trainer.fit(cli.model, cli.datamodule)
cli.trainer.callbacks = [
    cb
    for cb in cli.trainer.callbacks
    if not isinstance(cb, StochasticWeightAveraging)
]
if cli.trainer.fast_dev_run:
    print("Fast dev run, skipping test")
    print(cli.trainer.logged_metrics)
else:
    cli.trainer.test(
        cli.model,
        cli.datamodule,
        ckpt_path=cli.trainer.checkpoint_callback.best_model_path,
    )
    wandb.finish()
    if "test/MeanSquaredError" not in cli.trainer.logged_metrics:
        mae = cli.trainer.logged_metrics["test/MeanAbsoluteError"].item()
        mape = cli.trainer.logged_metrics["test/MeanAbsolutePercentageError"].item()
        rmse = cli.trainer.logged_metrics["test/RootMeanSquaredError"].item()
        results = {"mae": mae, "mape": mape, "rmse": rmse}
    else:
        mse_test = cli.trainer.logged_metrics["test/MeanSquaredError"].item()
        mae_test = cli.trainer.logged_metrics["test/MeanAbsoluteError"].item()
        results = {"mse_test": mse_test, "mae_test": mae_test}
    print(results)


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: Setting scheduler to Warm-Up
INFO:lightning.pytorch.utilities.rank_zero:Setting scheduler to Warm-Up
INFO: Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:lightning.pytorch.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable Li

train 12089


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


val 1661


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ criterion     │ L1Loss           │      0 │ train │     0 │
│ 1 │ train_metrics │ MetricCollection │      0 │ train │     0 │
│ 2 │ val_metrics   │ MetricCollection │      0 │ train │     0 │
│ 3 │ test_metrics  │ MetricCollection │      0 │ train │     0 │
│ 4 │ model         │ Transformixer    │ 25.4 M │ train │     0 │
└───┴───────────────┴──────────────────┴────────┴───────┴───────┘

...

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at outputs/aex28zcl/checkpoints/epoch=9-val_loss=0.24.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at outputs/aex28zcl/checkpoints/epoch=9-val_loss=0.24.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  test/MeanAbsoluteError   │    0.2557980418205261     │
│   test/MeanSquaredError   │    0.40676528215408325    │
└───────────────────────────┴───────────────────────────┘

epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
lr-Adam,███▇▅▃▁▁▁▁
test/MeanAbsoluteError,▁
test/MeanSquaredError,▁
train/MeanAbsoluteError,█▄▃▃▂▂▁▁▁▁
train/MeanSquaredError,█▄▃▂▂▁▁▁▁▁
train/loss_epoch,█▄▃▃▂▂▁▁▁▁
train/loss_step,█▅▆▅▄▃▃▄▃▂▃▂▃▄▃▂▂▄▃▂▁▃▁▂▂▂▂▂▂▃▂▂▁▁▂▃▁▂▂▂
trainer/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇██
val/MeanAbsoluteError,█▅▄▃▂▂▁▁▁▁
+2,...


{'mse_test': 0.40676528215408325, 'mae_test': 0.2557980418205261}
